# DEEP R + W1/W2 growth + hidden-unit growth on multi-MNIST 2-task

Same multi-MNIST 2-task / 1-hidden-LTU setup as
`deep_r_2layer_grow_io.ipynb`, but now hidden units themselves can be
created and destroyed.

- Hidden-unit slots are padded to `MAX_UNITS` with an `active` mask. Initial
  `N_HIDDEN` slots are populated with the same Kaiming-uniform init as the
  parent notebook.
- **W1 (incoming)**: prune always-on (sign-flip → deactivate). New W1
  entries grow from a shared connection budget.
- **W2 (outgoing)**: prune and grow gated by per-unit age — both fire once
  `age >= output_age_threshold` (default 10k).
- **Unit death**: when a W2 prune leaves a hidden unit with zero outgoing
  connections, that unit dies — its row in W2 and column in W1 are zeroed
  and the slot becomes free.
- **Generation budgets**: every connection that is pruned (W1, W2, or W1
  killed by a dying unit) adds 0.5 to a connection-budget pool and 0.5 to a
  unit-budget pool. These accumulate (fractions are fine).
  - Each step, draw up to `max_conn_gen_per_step` connections from a single
    shared pool of empty (input → active-unit) and (active-unit → output)
    slots. Sampling is uniform over slots, so the much larger W1 pool tends
    to absorb most of the conn budget.
  - When the unit budget reaches `INPUT_FANIN + 1`, spawn a new unit into
    the first free slot: pick `INPUT_FANIN` random inputs and one random
    output, set ages to 0.

## Setup

In [11]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from phd.jax_core.models import ltu

pio.templates.default = 'plotly'

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# Hidden-unit layout
N_HIDDEN = 20                                     # initial active hidden units
MAX_UNITS = 4 * N_HIDDEN                          # padded slot count
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM
INPUT_FANIN = 128                                  # initial active input connections per unit

assert N_HIDDEN % OUTPUT_DIM == 0, "N_HIDDEN must be divisible by OUTPUT_DIM"
assert MAX_UNITS >= N_HIDDEN

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  MAX_UNITS={MAX_UNITS}'
      f'  OUTPUT_DIM={OUTPUT_DIM}  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}'
      f'  INPUT_FANIN={INPUT_FANIN}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=20  MAX_UNITS=80  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=1  INPUT_FANIN=128


## Data

In [12]:
def load_data():
    """MNIST standardized per-pixel."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, '   labels:', labels.shape)

images: (60000, 784)    labels: (60000,)


## Architecture

Forward: `z1 = x @ (W1 * M1); h = ltu(z1); logits = h @ (W2 * M2)`. Per-task
softmax CE summed over 2 tasks. No biases.

In [13]:
def forward(W1, M1, W2, M2, x):
    z1 = x @ (W1 * M1)                              # (N_HIDDEN,)
    h = ltu(z1)                                     # binary {0,1} forward, sigmoid-STE backward
    logits = h @ (W2 * M2)                          # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W1, M1, W2, M2, x, y):
    logits, _, _ = forward(W1, M1, W2, M2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_pt, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init

Per hidden unit: pick `INPUT_FANIN` random input pixels; one-hot to its assigned
output. Weights at active entries are Kaiming-uniform with the appropriate fan-in.
Future extensions (more outgoing connections, feature growth) just modify these
masks before passing them to the train functions.

In [14]:
def init_2layer_ltu(seed=0, max_units=MAX_UNITS, n_initial=N_HIDDEN, input_fanin=INPUT_FANIN):
    """Init padded W1/M1/W2/M2 plus unit_active mask and unit_task vector.

    First `n_initial` slots populated like the parent notebook (Kaiming init,
    one-hot routing to outputs). Remaining slots zero / inactive."""
    k = jax.random.key(seed)
    k_m1, k_w1, k_w2 = jax.random.split(k, 3)

    keys = jax.random.split(k_m1, n_initial)
    def per_unit(key):
        noise = jax.random.uniform(key, (INPUT_DIM,))
        idx = jnp.argsort(-noise)[:input_fanin]
        return jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[idx].set(1)
    M1_active = jax.vmap(per_unit)(keys).T                         # (IN, n_initial)

    w1_bound = jnp.sqrt(3.0 / float(input_fanin))
    W1_active = jax.random.uniform(k_w1, (INPUT_DIM, n_initial),
                                    minval=-w1_bound, maxval=w1_bound) * M1_active

    hidden_per_output = n_initial // OUTPUT_DIM
    h2o = jnp.arange(n_initial) // hidden_per_output               # (n_initial,)
    M2_active = jax.nn.one_hot(h2o, OUTPUT_DIM, dtype=jnp.int32)   # (n_initial, OUT)
    w2_bound = jnp.sqrt(3.0 / float(hidden_per_output))
    W2_active = jax.random.uniform(k_w2, (n_initial, OUTPUT_DIM),
                                    minval=-w2_bound, maxval=w2_bound) * M2_active

    # Pad to max_units
    M1 = jnp.zeros((INPUT_DIM, max_units), dtype=jnp.int32).at[:, :n_initial].set(M1_active)
    W1 = jnp.zeros((INPUT_DIM, max_units), dtype=jnp.float32).at[:, :n_initial].set(W1_active)
    M2 = jnp.zeros((max_units, OUTPUT_DIM), dtype=jnp.int32).at[:n_initial, :].set(M2_active)
    W2 = jnp.zeros((max_units, OUTPUT_DIM), dtype=jnp.float32).at[:n_initial, :].set(W2_active)

    unit_active = jnp.zeros(max_units, dtype=jnp.int32).at[:n_initial].set(1)
    initial_unit_task = (h2o // NUM_CLASSES).astype(jnp.int32)     # (n_initial,)
    unit_task = jnp.zeros(max_units, dtype=jnp.int32).at[:n_initial].set(initial_unit_task)
    return W1, M1, W2, M2, unit_active, unit_task


_W1, _M1, _W2, _M2, _UA, _UT = init_2layer_ltu(seed=0)
print(f'Active units: {int(_UA.sum())} / {MAX_UNITS}')
print(f'M1 active per active unit: min={int(_M1.sum(0)[:N_HIDDEN].min())}'
      f' mean={float(_M1.sum(0)[:N_HIDDEN].mean())}'
      f' max={int(_M1.sum(0)[:N_HIDDEN].max())}')
print(f'M2 active per active unit: min={int(_M2.sum(1)[:N_HIDDEN].min())}'
      f' max={int(_M2.sum(1)[:N_HIDDEN].max())} (should both be 1)')

Active units: 20 / 80
M1 active per active unit: min=128 mean=128.0 max=128
M2 active per active unit: min=1 max=1 (should both be 1)


## DEEP R training (W1 always prunes; W2 age-gated; unit death + birth)

Per step:

1. Compute loss/grads on the (padded) W1, W2.
2. DEEP R update on each layer (then re-mask to keep inactive slots at 0).
3. **W1 deact (always)**: deactivate any sign-flipped active W1 entry.
4. **W2 deact (age-gated)**: same for W2, but only for units with
   `age >= output_age_threshold`.
5. **Unit death**: any unit that just lost its last W2 entry has its
   remaining W1 entries cleared and its slot freed (`unit_active = 0`).
6. **Budget update**: total prunings this step (W1 deact + W2 deact +
   W1 from dying units). `unit_budget_fraction` of every prune feeds the
   unit budget; the remaining `1 - unit_budget_fraction` feeds the
   connection budget.
7. **Connection generation (per-unit pool)**: each active unit contributes
   `INPUT_FANIN - active_W1[h]` slots to the W1 side of the pool and
   `OUTPUT_DIM - active_W2[h]` slots to the W2 side (W2 only if age-gate
   passes). Picks are drawn uniformly from this combined pool — so the
   relative chance of W1 vs W2 reflects per-unit free capacity, not raw
   matrix size. We sample up to `max_conn_gen_per_step` per step while
   `conn_budget >= 1` AND total active is below `max_active_connections`.
8. **Unit spawn**: if `unit_budget >= INPUT_FANIN + 1`, a free slot
   exists, AND adding `INPUT_FANIN + 1` connections still fits under
   `max_active_connections`, fill the slot with `INPUT_FANIN` random
   input connections and one random output connection. Reset its age to 0.

In [15]:
def train_deep_r(W1_init, M1_init, W2_init, M2_init,
                 unit_active_init, unit_task_init,
                 images, labels, *,
                 lr=2**-7,
                 l1=1e-4,
                 temperature=1e-7,
                 n_steps=500_000,
                 snapshot_every=2_000,
                 permute_period=0,
                 output_age_threshold=10_000,
                 output_max_gen_age=None,
                 input_init_magnitude=1e-3,
                 output_init_magnitude=1e-3,
                 max_conn_gen_per_step=1,
                 unit_budget_fraction=0.5,
                 max_active_connections=INPUT_FANIN * N_HIDDEN + N_HIDDEN,
                 gen_stop_step=None,
                 unit_gen_start_step=0,
                 unit_gen_stop_step=None,
                 output_reset_step=None,
                 seed=0):
    """DEEP R with shared conn-budget growth and hidden-unit creation/death.

    `unit_budget_fraction`: fraction of every prune that feeds the unit
        budget (the rest goes to the connection budget). Default 0.5.
    `max_active_connections`: cap on total active W1 + W2 entries. Once
        reached, no new connections are generated and no new units are
        spawned (a spawn would add INPUT_FANIN + 1 connections, which would
        push past the cap). Default = initial active count.
    `gen_stop_step`: optional global cutoff. Once `t >= gen_stop_step`,
        both connection generation and unit spawn are turned off. Pruning
        (W1 always, W2 age-gated) continues unaffected. Budgets keep
        accumulating but are never spent. None = never stop generation.
    `unit_gen_start_step` / `unit_gen_stop_step`: optional global window
        gating *unit spawning only* (does not affect connection generation).
        Spawns fire only when `unit_gen_start_step <= t < unit_gen_stop_step`.
        Defaults: start=0, stop=None (no upper bound). Connection generation
        is unaffected by these.
    `output_reset_step`: optional one-shot reset. When `t == output_reset_step`
        at the start of a step, all W2 weights are zeroed and all unit ages
        are reset to 0 (M2 masks are preserved). After the reset, training
        continues normally; the age reset gives each unit a fresh
        `output_age_threshold` window before W2 pruning re-engages.
    Connection sampling is over a *per-unit* free-capacity pool: each
    active unit has at most INPUT_FANIN incoming and OUTPUT_DIM outgoing
    slots. The W1 pool weight is sum_h (INPUT_FANIN - active_W1[h]) and
    the W2 pool weight is sum_h (OUTPUT_DIM - active_W2[h]) — both over
    eligible units. Picks are drawn uniformly from this combined pool.
    Newly-spawned units (cost INPUT_FANIN + 1 from unit_budget) get
    Kaiming-uniform input weights matching `init_2layer_ltu` and a single
    output connection at exactly 0 (mask=1, weight=0)."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    age_init = jnp.zeros(MAX_UNITS, dtype=jnp.int32)

    input_task_arr  = jnp.arange(INPUT_DIM) // INPUT_PER_TASK    # (IN,)
    output_task_arr = jnp.arange(OUTPUT_DIM) // NUM_CLASSES      # (OUT,)
    n_w1_pool = MAX_UNITS * INPUT_FANIN                           # per-unit W1 pool size
    n_w2_pool = MAX_UNITS * OUTPUT_DIM                            # per-unit W2 pool size
    conn_frac = float(1.0 - unit_budget_fraction)
    unit_frac = float(unit_budget_fraction)
    spawn_w1_bound = float((3.0 / float(INPUT_FANIN)) ** 0.5)    # Kaiming-uniform bound

    # Pre-seed the budgets with any initial slack between the active count
    # and max_active_connections, split by unit_budget_fraction. This lets
    # generation/spawning fire on step 0 when starting under-capacity (e.g.
    # max_active_connections raised above the initialization size), without
    # needing prunes first.
    initial_active = (jnp.sum(M1_init.astype(jnp.int32))
                      + jnp.sum(M2_init.astype(jnp.int32))).astype(jnp.float32)
    initial_excess = jnp.maximum(jnp.float32(max_active_connections) - initial_active, 0.0)
    conn_budget_init = conn_frac * initial_excess
    unit_budget_init = unit_frac * initial_excess

    init_carry = (
        W1_init, M1_init.astype(jnp.int32),
        W2_init, M2_init.astype(jnp.int32),
        unit_active_init.astype(jnp.int32),
        unit_task_init.astype(jnp.int32),
        age_init,
        perm0_init, perm1_init,
        jnp.array(0, dtype=jnp.int32),                           # t
        conn_budget_init,                                        # conn_budget (seeded with initial slack)
        unit_budget_init,                                        # unit_budget (seeded with initial slack)
        jnp.array(0, dtype=jnp.int32),                           # cum_d1w
        jnp.array(0, dtype=jnp.int32),                           # cum_d1c
        jnp.array(0, dtype=jnp.int32),                           # cum_d2w
        jnp.array(0, dtype=jnp.int32),                           # cum_d2c
        jnp.array(0, dtype=jnp.int32),                           # cum_units_gen
        jnp.array(0, dtype=jnp.int32),                           # cum_units_pruned
    )
    noise_scale = jnp.sqrt(2.0 * lr * temperature)

    def step_fn(carry, key):
        (W1, M1, W2, M2, unit_active, unit_task, age,
         perm0, perm1, t,
         conn_budget, unit_budget,
         cum_d1w, cum_d1c, cum_d2w, cum_d2c,
         cum_ug, cum_up) = carry
        keys = jax.random.split(key, 11)
        (data_key, n1_key, n2_key, perm_key,
         conn_w1_key, conn_w2_key, conn_sign_key,
         spawn_in_key, spawn_in_w_key,
         spawn_out_key, _unused_key) = keys

        # Optional one-shot output reset at exactly t == output_reset_step.
        # Zeros W2 (M2 mask preserved) and resets all ages to 0 so the
        # next output_age_threshold steps are W2-prune-safe.
        if output_reset_step is not None:
            do_reset = t == jnp.int32(output_reset_step)
            W2 = jnp.where(do_reset, jnp.zeros_like(W2), W2)
            age = jnp.where(do_reset, jnp.zeros_like(age), age)

        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        def _loss(W1_, W2_):
            return loss_fn(W1_, M1, W2_, M2, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)

        s1 = jnp.sign(W1)
        s2 = jnp.sign(W2)
        n1 = jax.random.normal(n1_key, W1.shape) * noise_scale
        n2 = jax.random.normal(n2_key, W2.shape) * noise_scale
        W1u = (W1 - lr * g1 - lr * l1 * s1 + n1) * M1
        W2u = (W2 - lr * g2 - lr * l1 * s2 + n2) * M2

        # === W1 deactivation (always on) ===
        deact1 = (jnp.sign(W1u) != s1) & (M1 == 1)
        d1_int = deact1.astype(jnp.int32)
        M1a = M1 * (1 - d1_int)
        W1a = W1u * M1a

        # === W2 deactivation (age-gated) ===
        elig_out_h = (age >= output_age_threshold) & (unit_active == 1)
        deact2 = ((jnp.sign(W2u) != s2) & (M2 == 1) & elig_out_h[:, None])
        d2_int = deact2.astype(jnp.int32)
        M2a = M2 * (1 - d2_int)
        W2a = W2u * M2a

        # === Unit death ===
        n_W2_before = M2.sum(axis=-1)
        n_W2_after  = M2a.sum(axis=-1)
        died = (unit_active == 1) & (n_W2_before > 0) & (n_W2_after == 0)
        died_int = died.astype(jnp.int32)
        # W1 entries killed by death:
        d3_int = M1a * died_int[None, :]                                # (IN, MAX_UNITS)

        # Within/cross task tallies
        same_task_ih = (input_task_arr[:, None] == unit_task[None, :])  # (IN, MAX_UNITS)
        same_task_ho = (unit_task[:, None] == output_task_arr[None, :]) # (MAX_UNITS, OUT)
        cum_d1w_new = cum_d1w + jnp.sum((d1_int + d3_int) * same_task_ih.astype(jnp.int32))
        cum_d1c_new = cum_d1c + jnp.sum((d1_int + d3_int) * (~same_task_ih).astype(jnp.int32))
        cum_d2w_new = cum_d2w + jnp.sum(d2_int * same_task_ho.astype(jnp.int32))
        cum_d2c_new = cum_d2c + jnp.sum(d2_int * (~same_task_ho).astype(jnp.int32))

        # Apply unit death
        M1b = M1a * (1 - died_int[None, :])
        W1b = W1a * M1b
        ua_b = unit_active * (1 - died_int)
        age_b = age * (1 - died_int)

        # Total prunings -> budgets (split by unit_budget_fraction)
        n_pruned = (jnp.sum(d1_int) + jnp.sum(d2_int) + jnp.sum(d3_int)).astype(jnp.float32)
        conn_budget_b = conn_budget + conn_frac * n_pruned
        unit_budget_b = unit_budget + unit_frac * n_pruned
        cum_up_new = cum_up + jnp.sum(died_int)

        # Total active connections after pruning (used for cap gating)
        n_active_total = jnp.sum(M1b) + jnp.sum(M2a)
        room = jnp.maximum(max_active_connections - n_active_total, 0)

        # Generation cutoff: if gen_stop_step is set and t has reached it,
        # disable both connection generation and unit spawn this step.
        if gen_stop_step is None:
            gen_active = jnp.array(True)
        else:
            gen_active = t < jnp.int32(gen_stop_step)

        # Unit-spawn window (independent of conn-gen gating).
        unit_gen_after_start = t >= jnp.int32(unit_gen_start_step)
        if unit_gen_stop_step is None:
            unit_gen_before_stop = jnp.array(True)
        else:
            unit_gen_before_stop = t < jnp.int32(unit_gen_stop_step)
        unit_gen_active = unit_gen_after_start & unit_gen_before_stop

        # === Connection generation (per-unit-cap-aware sampling) ===
        # Per-unit free incoming capacity. Each active unit can hold at most
        # INPUT_FANIN active W1 entries.
        free_w1_per_unit = jnp.where(
            ua_b == 1,
            jnp.maximum(INPUT_FANIN - M1b.sum(axis=0), 0),
            0,
        )                                                                # (MAX_UNITS,)

        # Score every empty (input, unit) cell, then per-unit pick the top
        # INPUT_FANIN by score and keep only the first free_w1_per_unit[h]
        # of them. This makes the per-unit W1 pool size exactly equal to
        # free_w1_per_unit[h] (rather than INPUT_DIM - active).
        rand_w1 = jax.random.uniform(conn_w1_key, (INPUT_DIM, MAX_UNITS))
        empty_w1 = (ua_b[None, :] == 1) & (M1b == 0)
        score_w1 = jnp.where(empty_w1, rand_w1, -1.0)                    # (IN, MAX_UNITS)
        top_w1_vals, top_w1_inputs = jax.lax.top_k(score_w1.T, INPUT_FANIN)  # (MAX_UNITS, INPUT_FANIN)
        positions_w1 = jnp.arange(INPUT_FANIN)
        eligible_w1 = (positions_w1[None, :] < free_w1_per_unit[:, None]) & (top_w1_vals > -0.5)
        w1_pool_score = jnp.where(eligible_w1, top_w1_vals, -1.0)        # (MAX_UNITS, INPUT_FANIN)

        # W2 pool: per-unit cap is OUTPUT_DIM (the matrix dimension itself),
        # so no extra masking beyond the empty + age gate.
        elig_out_h_b = (age_b >= output_age_threshold) & (ua_b == 1)
        if output_max_gen_age is None:
            grow_out_h = elig_out_h_b
        else:
            grow_out_h = elig_out_h_b & (age_b < output_max_gen_age)
        rand_w2 = jax.random.uniform(conn_w2_key, (MAX_UNITS, OUTPUT_DIM))
        empty_w2 = grow_out_h[:, None] & (M2a == 0)
        w2_pool_score = jnp.where(empty_w2, rand_w2, -1.0)               # (MAX_UNITS, OUT)

        flat_combined = jnp.concatenate([
            w1_pool_score.reshape(-1),                                    # MAX_UNITS * INPUT_FANIN
            w2_pool_score.reshape(-1),                                    # MAX_UNITS * OUTPUT_DIM
        ])
        top_score, top_idx = jax.lax.top_k(flat_combined, max_conn_gen_per_step)

        # n_to_add capped by: budget, per-step limit, remaining cap room,
        # and the global gen cutoff.
        n_to_add = jnp.minimum(jnp.floor(conn_budget_b).astype(jnp.int32),
                                max_conn_gen_per_step)
        n_to_add = jnp.minimum(n_to_add, room)
        n_to_add = jnp.where(gen_active, n_to_add, jnp.int32(0))
        positions = jnp.arange(max_conn_gen_per_step)
        will_activate = (positions < n_to_add) & (top_score > -0.5)

        signs = jax.random.choice(conn_sign_key, jnp.array([-1.0, 1.0]),
                                   shape=(max_conn_gen_per_step,))
        is_w1 = top_idx < n_w1_pool
        do_w1 = will_activate & is_w1
        do_w2 = will_activate & ~is_w1

        # Decompose flat indices (clip to safe range so the off-branch indexing
        # doesn't crash; the corresponding `do_*` mask zeroes its effect).
        idx_w1_safe = jnp.minimum(top_idx, n_w1_pool - 1)
        h_w1 = idx_w1_safe // INPUT_FANIN
        p_w1 = idx_w1_safe % INPUT_FANIN
        i_w1 = top_w1_inputs[h_w1, p_w1]                                 # actual input dim
        idx_w2 = jnp.minimum(jnp.maximum(top_idx - n_w1_pool, 0), n_w2_pool - 1)
        h_w2 = idx_w2 // OUTPUT_DIM
        o_w2 = idx_w2 % OUTPUT_DIM

        flat_M1 = M1b.reshape(-1)
        flat_W1 = W1b.reshape(-1)
        w1_flat = i_w1 * MAX_UNITS + h_w1
        flat_M1 = flat_M1.at[w1_flat].set(jnp.where(do_w1, 1, flat_M1[w1_flat]))
        flat_W1 = flat_W1.at[w1_flat].set(
            jnp.where(do_w1, signs * input_init_magnitude, flat_W1[w1_flat]))
        M1c = flat_M1.reshape(M1b.shape)
        W1c = flat_W1.reshape(W1b.shape)

        flat_M2 = M2a.reshape(-1)
        flat_W2 = W2a.reshape(-1)
        w2_flat = h_w2 * OUTPUT_DIM + o_w2
        flat_M2 = flat_M2.at[w2_flat].set(jnp.where(do_w2, 1, flat_M2[w2_flat]))
        flat_W2 = flat_W2.at[w2_flat].set(
            jnp.where(do_w2, signs * output_init_magnitude, flat_W2[w2_flat]))
        M2b = flat_M2.reshape(M2a.shape)
        W2b = flat_W2.reshape(W2a.shape)

        n_generated = jnp.sum(will_activate.astype(jnp.float32))
        conn_budget_c = conn_budget_b - n_generated

        # === Unit spawn ===
        # Inputs: Kaiming-uniform (matching init_2layer_ltu).
        # Output weight: 0 at the chosen output (mask=1, weight=0). The unit
        # learns its outgoing magnitude/sign under DEEP R dynamics during its
        # first output_age_threshold steps before it becomes prune-eligible.
        # A spawn adds INPUT_FANIN + 1 connections, so we only spawn if those
        # would still fit under max_active_connections.
        n_active_total_post_gen = n_active_total + n_generated.astype(jnp.int32)
        spawn_fits = (n_active_total_post_gen + (INPUT_FANIN + 1)
                       <= max_active_connections)
        any_free = jnp.any(ua_b == 0)
        slot = jnp.argmin(ua_b).astype(jnp.int32)
        will_spawn = ((unit_budget_b >= float(INPUT_FANIN + 1)) & any_free
                       & spawn_fits & gen_active & unit_gen_active)
        will_spawn_f = will_spawn.astype(jnp.float32)
        will_spawn_i = will_spawn.astype(jnp.int32)

        in_scores = jax.random.uniform(spawn_in_key, (INPUT_DIM,))
        _, in_idx = jax.lax.top_k(in_scores, INPUT_FANIN)
        new_M1_col = jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[in_idx].set(1)
        in_vals = jax.random.uniform(spawn_in_w_key, (INPUT_FANIN,),
                                      minval=-spawn_w1_bound,
                                      maxval=spawn_w1_bound)
        new_W1_col = jnp.zeros(INPUT_DIM, dtype=jnp.float32).at[in_idx].set(in_vals)

        out_idx = jax.random.randint(spawn_out_key, (), 0, OUTPUT_DIM)
        new_M2_row = jnp.zeros(OUTPUT_DIM, dtype=jnp.int32).at[out_idx].set(1)
        new_W2_row = jnp.zeros(OUTPUT_DIM, dtype=jnp.float32)         # W2 starts at 0

        # Conditionally write into slot
        cur_M1_col = M1c[:, slot]
        cur_W1_col = W1c[:, slot]
        cur_M2_row = M2b[slot, :]
        cur_W2_row = W2b[slot, :]
        M1d = M1c.at[:, slot].set(jnp.where(will_spawn, new_M1_col, cur_M1_col))
        W1d = W1c.at[:, slot].set(jnp.where(will_spawn, new_W1_col, cur_W1_col))
        M2d = M2b.at[slot, :].set(jnp.where(will_spawn, new_M2_row, cur_M2_row))
        W2d = W2b.at[slot, :].set(jnp.where(will_spawn, new_W2_row, cur_W2_row))

        ua_c = ua_b.at[slot].set(jnp.where(will_spawn, jnp.int32(1), ua_b[slot]))
        new_task = (out_idx // NUM_CLASSES).astype(jnp.int32)
        unit_task_b = unit_task.at[slot].set(jnp.where(will_spawn, new_task, unit_task[slot]))
        age_c = age_b.at[slot].set(jnp.where(will_spawn, jnp.int32(0), age_b[slot]))

        unit_budget_c = unit_budget_b - will_spawn_f * float(INPUT_FANIN + 1)
        cum_ug_new = cum_ug + will_spawn_i

        # Increment age for active units only
        age_final = age_c + ua_c

        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0_new = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1_new = jnp.where(should_perm & (which == 1), new_perm, perm1)
        else:
            perm0_new = perm0
            perm1_new = perm1

        new_carry = (
            W1d, M1d, W2d, M2d, ua_c, unit_task_b, age_final,
            perm0_new, perm1_new, t_next,
            conn_budget_c, unit_budget_c,
            cum_d1w_new, cum_d1c_new, cum_d2w_new, cum_d2c_new,
            cum_ug_new, cum_up_new,
        )
        return new_carry, loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        (W1, M1, W2, M2, ua, ut, age,
         _p0, _p1, t,
         conn_b, unit_b,
         cum_d1w, cum_d1c, cum_d2w, cum_d2c,
         cum_ug, cum_up) = carry
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            M1=M1, M2=M2,
            unit_active=ua, unit_task=ut,
            n_active_units=ua.sum(),
            n_active_W1=M1.sum(),
            n_active_W2=M2.sum(),
            conn_budget=conn_b,
            unit_budget=unit_b,
            cum_deact_W1_within=cum_d1w,
            cum_deact_W1_cross=cum_d1c,
            cum_deact_W2_within=cum_d2w,
            cum_deact_W2_cross=cum_d2c,
            cum_units_generated=cum_ug,
            cum_units_pruned=cum_up,
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_M1'] = jax.device_get(final_carry[1])
    snaps['final_W2'] = jax.device_get(final_carry[2])
    snaps['final_M2'] = jax.device_get(final_carry[3])
    snaps['final_unit_active'] = jax.device_get(final_carry[4])
    snaps['final_unit_task']   = jax.device_get(final_carry[5])
    return snaps


## Baseline (no-prune)

Same init, plain SGD, masks frozen at the initial values. Loss curve is the
no-prune reference.

In [16]:
def train_baseline(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                   lr=2**-7,
                   n_steps=500_000,
                   snapshot_every=2_000,
                   permute_period=0,
                   seed=0):
    """Plain SGD with frozen masks. No L1, no noise, no deactivation.

    `permute_period`: 0 = stationary. >0 = same non-stationary mechanism as
    train_deep_r so the loss curves are directly comparable."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    init_carry = (W1_init, W2_init, perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, W2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1_init, W2_, M2_init, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)
        W1 = (W1 - lr * g1) * M1_init
        W2 = (W2 - lr * g2) * M2_init
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)
        return (W1, W2, perm0, perm1, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, W2, _p0, _p1, t = carry
        return carry, dict(step=t, avg_loss=losses.mean())

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    return snaps

## Run

The DEEP R run takes longer because of the noise updates and the per-chunk
mask snapshots. Reduce `n_steps` for fast iteration.

In [17]:
# Shared init (same seed -> same starting topology and weights for both runs).
W1_init, M1_init, W2_init, M2_init, ua_init, ut_init = init_2layer_ltu(seed=0)

# DEEP R hyperparameters - edit freely.
DEEP_R_CONFIG = dict(
    lr=2**-6,
    l1=1e-4,
    temperature=1e-7,
    n_steps=1_000_000,
    snapshot_every=2_000,
    permute_period=0,
    output_age_threshold=10_000,           # per-unit age before W2 prunes/grows
    output_max_gen_age=None,                # optional W2 grow cutoff
    input_init_magnitude=1e-3,
    output_init_magnitude=1e-3,
    max_conn_gen_per_step=8,                # cap on new connections per step
    unit_budget_fraction=0.05,               # fraction of prunes that feeds the unit budget
    max_active_connections=INPUT_FANIN * N_HIDDEN + OUTPUT_DIM * N_HIDDEN / 2 // 2,  # cap on total active W1+W2
    gen_stop_step=None,                     # global cutoff: t >= this -> no generation/spawn (pruning still on)
    unit_gen_start_step=0,                  # spawn units only when t >= this (conn-gen unaffected)
    unit_gen_stop_step=500_000,                # spawn units only when t < this (conn-gen unaffected)
    output_reset_step=500_000,                 # at this step: W2=0 and all ages=0 (M2 mask preserved)
    seed=0,
)

train_deep_r_jit = jax.jit(
    train_deep_r,
    static_argnames=('lr', 'l1', 'temperature', 'n_steps', 'snapshot_every',
                     'permute_period',
                     'output_age_threshold', 'output_max_gen_age',
                     'input_init_magnitude', 'output_init_magnitude',
                     'max_conn_gen_per_step',
                     'unit_budget_fraction',
                     'max_active_connections',
                     'gen_stop_step',
                     'unit_gen_start_step', 'unit_gen_stop_step',
                     'output_reset_step',
                     'seed'),
)
train_baseline_jit = jax.jit(
    train_baseline,
    static_argnames=('lr', 'n_steps', 'snapshot_every', 'permute_period', 'seed'),
)

print('Running DEEP R...')
deep_r_snaps = train_deep_r_jit(W1_init, M1_init, W2_init, M2_init,
                                ua_init, ut_init, images, labels, **DEEP_R_CONFIG)
deep_r_snaps['M1_init'] = np.asarray(M1_init)
deep_r_snaps['M2_init'] = np.asarray(M2_init)
deep_r_snaps['unit_active_init'] = np.asarray(ua_init)
deep_r_snaps['unit_task_init']   = np.asarray(ut_init)
print(f'  final loss: {float(deep_r_snaps["avg_loss"][-1]):.4f}')
print(f'  active units: {int(deep_r_snaps["n_active_units"][-1])} / {MAX_UNITS}'
      f' (init {int(ua_init.sum())})')
print(f'  W1 active: {int(deep_r_snaps["n_active_W1"][-1])}'
      f' (init {int(M1_init.sum())}, max possible {INPUT_DIM * MAX_UNITS})')
print(f'  W2 active: {int(deep_r_snaps["n_active_W2"][-1])}'
      f' (init {int(M2_init.sum())}, max possible {MAX_UNITS * OUTPUT_DIM})')
print(f'  total active conns: {int(deep_r_snaps["n_active_W1"][-1]) + int(deep_r_snaps["n_active_W2"][-1])}'
      f' / {DEEP_R_CONFIG["max_active_connections"]} cap')
print(f'  conn_budget: {float(deep_r_snaps["conn_budget"][-1]):.2f}'
      f'   unit_budget: {float(deep_r_snaps["unit_budget"][-1]):.2f}')
print(f'  units generated: {int(deep_r_snaps["cum_units_generated"][-1])}'
      f'   units pruned: {int(deep_r_snaps["cum_units_pruned"][-1])}')

print('\nRunning baseline (no-prune, no-grow)...')
baseline_snaps = train_baseline_jit(W1_init, M1_init, W2_init, M2_init, images, labels,
                                     lr=DEEP_R_CONFIG['lr'],
                                     n_steps=DEEP_R_CONFIG['n_steps'],
                                     snapshot_every=DEEP_R_CONFIG['snapshot_every'],
                                     permute_period=DEEP_R_CONFIG['permute_period'],
                                     seed=DEEP_R_CONFIG['seed'])
print(f'  final loss: {float(baseline_snaps["avg_loss"][-1]):.4f}')


Running DEEP R...
  final loss: 0.3631
  active units: 70 / 80 (init 20)
  W1 active: 1413 (init 2560, max possible 125440)
  W2 active: 133 (init 20, max possible 1600)
  total active conns: 1546 / 2660.0 cap
  conn_budget: 0.80   unit_budget: 1113.25
  units generated: 88   units pruned: 38

Running baseline (no-prune, no-grow)...
  final loss: 1.7015


## Metrics — connection and unit growth/death tracking

For each layer (W1, W2): track active counts split by within/cross task
(using each snapshot's `unit_task` since slot identities can change), and
the cumulative within/cross prunings.

For units: active count over time, cumulative spawned and killed,
plus the running connection-budget and unit-budget accumulators.

In [18]:
INPUT_TASK  = np.arange(INPUT_DIM)  // INPUT_PER_TASK              # (IN,)
OUTPUT_TASK = np.arange(OUTPUT_DIM) // NUM_CLASSES                # (OUT,)


def compute_w1_metrics(snaps):
    """W1 active counts split by within/cross task using per-snapshot
    unit_task; plus cumulative within/cross deactivations."""
    M1s = np.asarray(snaps['M1']).astype(np.int32)             # (n_chunks, IN, MAX_UNITS)
    UTs = np.asarray(snaps['unit_task']).astype(np.int32)      # (n_chunks, MAX_UNITS)
    same = (INPUT_TASK[None, :, None] == UTs[:, None, :])      # (n_chunks, IN, MAX_UNITS)
    same = same.astype(M1s.dtype)
    active_total  = M1s.sum(axis=(1, 2))
    active_within = (M1s * same).sum(axis=(1, 2))
    active_cross  = active_total - active_within
    return dict(
        steps=np.asarray(snaps['step']),
        active_total=active_total,
        active_within=active_within,
        active_cross=active_cross,
        cum_deact_within=np.asarray(snaps['cum_deact_W1_within']),
        cum_deact_cross=np.asarray(snaps['cum_deact_W1_cross']),
    )


def compute_w2_metrics(snaps):
    M2s = np.asarray(snaps['M2']).astype(np.int32)             # (n_chunks, MAX_UNITS, OUT)
    UTs = np.asarray(snaps['unit_task']).astype(np.int32)      # (n_chunks, MAX_UNITS)
    same = (UTs[:, :, None] == OUTPUT_TASK[None, None, :])     # (n_chunks, MAX_UNITS, OUT)
    same = same.astype(M2s.dtype)
    active_total  = M2s.sum(axis=(1, 2))
    active_within = (M2s * same).sum(axis=(1, 2))
    active_cross  = active_total - active_within
    return dict(
        steps=np.asarray(snaps['step']),
        active_total=active_total,
        active_within=active_within,
        active_cross=active_cross,
        cum_deact_within=np.asarray(snaps['cum_deact_W2_within']),
        cum_deact_cross=np.asarray(snaps['cum_deact_W2_cross']),
    )


def compute_unit_metrics(snaps):
    return dict(
        steps=np.asarray(snaps['step']),
        n_active_units=np.asarray(snaps['n_active_units']),
        cum_generated=np.asarray(snaps['cum_units_generated']),
        cum_pruned=np.asarray(snaps['cum_units_pruned']),
        conn_budget=np.asarray(snaps['conn_budget']),
        unit_budget=np.asarray(snaps['unit_budget']),
    )


w1_metrics = compute_w1_metrics(deep_r_snaps)
w2_metrics = compute_w2_metrics(deep_r_snaps)
u_metrics  = compute_unit_metrics(deep_r_snaps)
print(f'final W1 active total / within / cross: '
      f'{int(w1_metrics["active_total"][-1])}'
      f' / {int(w1_metrics["active_within"][-1])}'
      f' / {int(w1_metrics["active_cross"][-1])}')
print(f'final W2 active total / within / cross: '
      f'{int(w2_metrics["active_total"][-1])}'
      f' / {int(w2_metrics["active_within"][-1])}'
      f' / {int(w2_metrics["active_cross"][-1])}')
print(f'cumulative W1 prunings within / cross: '
      f'{int(w1_metrics["cum_deact_within"][-1])}'
      f' / {int(w1_metrics["cum_deact_cross"][-1])}')
print(f'cumulative W2 prunings within / cross: '
      f'{int(w2_metrics["cum_deact_within"][-1])}'
      f' / {int(w2_metrics["cum_deact_cross"][-1])}')
print(f'units generated / pruned: '
      f'{int(u_metrics["cum_generated"][-1])} / {int(u_metrics["cum_pruned"][-1])}')


final W1 active total / within / cross: 1413 / 1390 / 23
final W2 active total / within / cross: 133 / 131 / 2
cumulative W1 prunings within / cross: 120777 / 125911
cumulative W2 prunings within / cross: 1194 / 1342
units generated / pruned: 88 / 38


## Plots

In [19]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red
GEN_COLOR    = '#2ca02c'   # green
PRUNE_COLOR  = '#9467bd'   # purple


def plot_loss(deep_r_snaps, baseline_snaps):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.asarray(deep_r_snaps['step']),
                             y=np.asarray(deep_r_snaps['avg_loss']),
                             mode='lines', name='DEEP R + unit growth'))
    fig.add_trace(go.Scatter(x=np.asarray(baseline_snaps['step']),
                             y=np.asarray(baseline_snaps['avg_loss']),
                             mode='lines', name='baseline (no prune/grow)',
                             line=dict(dash='dot')))
    fig.update_layout(title='Loss over training',
                      xaxis_title='step', yaxis_title='mean loss over snapshot',
                      width=900, height=420)
    fig.show()
    return fig


def _plot_active_per_unit(metrics, layer_name):
    """Active connections split by within/cross task plus total."""
    steps = metrics['steps']
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=metrics['active_within'],
                             mode='lines', name='within-task active',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=metrics['active_cross'],
                             mode='lines', name='cross-task active',
                             line=dict(color=CROSS_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=metrics['active_total'],
                             mode='lines', name='total active',
                             line=dict(dash='dot')))
    fig.update_layout(title=f'{layer_name}: total active connections',
                      xaxis_title='step', yaxis_title='# active',
                      width=900, height=420)
    fig.show()
    return fig


def plot_w1_active(w1_metrics):
    return _plot_active_per_unit(w1_metrics, 'W1 (incoming)')


def plot_w2_active(w2_metrics):
    return _plot_active_per_unit(w2_metrics, 'W2 (outgoing)')


def _plot_cumulative_pruned(metrics, layer_name):
    steps = metrics['steps']
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=metrics['cum_deact_within'],
                             mode='lines', name='within-task pruned (cumulative)',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=metrics['cum_deact_cross'],
                             mode='lines', name='cross-task pruned (cumulative)',
                             line=dict(color=CROSS_COLOR)))
    fig.update_layout(title=f'{layer_name}: cumulative connections pruned',
                      xaxis_title='step', yaxis_title='# cumulative deactivations',
                      width=900, height=420)
    fig.show()
    return fig


def plot_w1_cumulative_pruned(w1_metrics):
    return _plot_cumulative_pruned(w1_metrics, 'W1 (incoming)')


def plot_w2_cumulative_pruned(w2_metrics):
    return _plot_cumulative_pruned(w2_metrics, 'W2 (outgoing)')


def plot_active_units(u_metrics):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=u_metrics['steps'], y=u_metrics['n_active_units'],
                             mode='lines', name='active hidden units'))
    fig.update_layout(title='Hidden units active over training',
                      xaxis_title='step', yaxis_title='# active units',
                      width=900, height=420)
    fig.show()
    return fig


def plot_unit_births_deaths(u_metrics):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=u_metrics['steps'], y=u_metrics['cum_generated'],
                             mode='lines', name='units generated (cumulative)',
                             line=dict(color=GEN_COLOR)))
    fig.add_trace(go.Scatter(x=u_metrics['steps'], y=u_metrics['cum_pruned'],
                             mode='lines', name='units pruned (cumulative)',
                             line=dict(color=PRUNE_COLOR)))
    fig.update_layout(title='Hidden units generated / pruned',
                      xaxis_title='step', yaxis_title='# units (cumulative)',
                      width=900, height=420)
    fig.show()
    return fig


def plot_budgets(u_metrics):
    fig = make_subplots(rows=1, cols=2, subplot_titles=('connection budget', 'unit budget'))
    fig.add_trace(go.Scatter(x=u_metrics['steps'], y=u_metrics['conn_budget'],
                             mode='lines', name='conn_budget',
                             line=dict(color=GEN_COLOR)), row=1, col=1)
    fig.add_trace(go.Scatter(x=u_metrics['steps'], y=u_metrics['unit_budget'],
                             mode='lines', name='unit_budget',
                             line=dict(color=PRUNE_COLOR)), row=1, col=2)
    fig.add_hline(y=INPUT_FANIN + 1, line_dash='dot',
                  annotation_text=f'spawn cost = {INPUT_FANIN + 1}',
                  row=1, col=2)
    fig.update_layout(title='Generation budgets over training',
                      width=1100, height=420, showlegend=False)
    fig.update_xaxes(title_text='step')
    fig.show()
    return fig


In [20]:
plot_loss(deep_r_snaps, baseline_snaps)
plot_active_units(u_metrics)
plot_unit_births_deaths(u_metrics)
plot_budgets(u_metrics)
plot_w1_active(w1_metrics)
plot_w2_active(w2_metrics)
plot_w1_cumulative_pruned(w1_metrics)
plot_w2_cumulative_pruned(w2_metrics);